# Cosmic Sort 2026 - v7 (v6 + rebuilt neural net)

**What is new compared with v6** (v6 scored 0.136 on the public leaderboard)

1. The neural net is rebuilt: per-feature **piecewise-linear embeddings**, **LayerNorm** after every hidden layer, widths 512 / 256 / 128, **AdamW with weight decay 1e-2** (weights only), a fixed 30-epoch OneCycle schedule, and **weight averaging (EMA)** instead of picking a best epoch on the validation fold.
2. The net uses the GPU if one exists (5 seeds), otherwise the CPU (2 seeds). It prints a train/validation log for seed 0 and a "best epoch" line for every fit.
3. Net checkpoints are saved as `MLP2` files, so v6 net checkpoints are never reused. LightGBM and XGBoost checkpoints are unchanged.
4. Features, LightGBM and XGBoost settings, the blend and the submission steps are identical to v6 (blur and fuzziness features included). Output file: `submission_v7.csv`.

Suggested order: `QUICK = True` once (code check), then the full run. The per-fold v5-vs-v6 table printed after the blend is the check that the gain is real.

In [ ]:
import importlib, itertools, json, os, subprocess, sys, time, glob, zipfile
from pathlib import Path

USE_CATBOOST = False    # True adds CatBoost: gain was only ~0.0003 in a test, but it is far slower
USE_NN       = True     # v6: add a small neural net (MLP) to the blend
USE_BLUR     = True     # v6: local-blur features (neighbour sizes on the sky, no labels used)
USE_FUZZ     = True     # v6: error-fuzziness features (error size relative to what the magnitude predicts)
RUN_ABLATION = False    # True = extra ~15 min check that ranks the feature groups on all 5 folds (LightGBM, lr 0.1)

pkgs = [('numpy','numpy'),('pandas','pandas'),('scipy','scipy'),('sklearn','scikit-learn'),
        ('lightgbm','lightgbm>=4.0'),('xgboost','xgboost>=2.0')]
if USE_CATBOOST:
    pkgs.append(('catboost','catboost>=1.2'))
if USE_NN:
    pkgs.append(('torch','torch'))          # already installed on Colab and Kaggle
for module, package in pkgs:
    try:
        importlib.import_module(module)
    except ImportError:
        print('Installing', package); subprocess.check_call([sys.executable,'-m','pip','install','-q',package])

import numpy as np, pandas as pd
from scipy.optimize import minimize, minimize_scalar
from scipy.spatial import cKDTree
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import QuantileTransformer
import lightgbm as lgb, xgboost as xgb
if USE_CATBOOST:
    from catboost import CatBoostClassifier, Pool
if USE_NN:
    import copy, torch, torch.nn as nn, torch.nn.functional as F

# ---------------- settings ----------------
METRIC   = 'plain'      # 'plain' = average over all objects (matches the Kaggle leaderboard)
                        # 'balanced' = rulebook wording (each class averaged first)
FOLDS    = 5
SEED     = 42
N_SEEDS  = 3            # seed bagging for LightGBM / XGBoost (1 = off)
NN_GPU     = True       # use a GPU for the net if one exists
DEVICE     = torch.device('cuda' if (NN_GPU and torch.cuda.is_available()) else 'cpu') if USE_NN else None
N_SEEDS_NN = 5 if (USE_NN and DEVICE.type == 'cuda') else 2     # seed bagging for the net
NN_EPOCHS, NN_BATCH = 30, 1024
NN_LR, NN_WD, NN_EMA = 2e-3, 1e-2, 0.998
NN_WIDTHS, NN_DROPS = (512, 256, 128), (0.25, 0.20, 0.10)
NN_BINS, NN_EMB = 12, 8
NN_VERSION = 'MLP2'     # checkpoint name for the net: old v6 'MLP' checkpoints are NOT reused
QUICK    = False        # True = 3,000 rows and 60 trees, only to check the code runs (writes no submission)
GPU      = False        # True on a GPU runtime: speeds up XGBoost and CatBoost
THREADS  = os.cpu_count() or 2
TAG      = 'v6'         # checkpoint prefix: v6 features differ from v5, so old ckpt files are NOT reused
CKPT_DIR = Path('ckpt'); CKPT_DIR.mkdir(exist_ok=True)
DRIVE_DIR = None        # e.g. '/content/drive/MyDrive/cosmic-sort' if your CSVs are in Google Drive
if QUICK:
    N_SEEDS = min(N_SEEDS, 2); N_SEEDS_NN = min(N_SEEDS_NN, 2); NN_EPOCHS = 3
if USE_NN:
    torch.set_num_threads(THREADS)
print('METRIC =', METRIC, '| threads =', THREADS, '| QUICK =', QUICK, '| N_SEEDS =', N_SEEDS,
      '| N_SEEDS_NN =', N_SEEDS_NN, '| device =', DEVICE, '| blur', USE_BLUR, '| fuzz', USE_FUZZ, '| NN', USE_NN)

## 1. Load the data (Kaggle, Colab upload, Drive, or local folder)

In [ ]:
BANDS = list('ugriz'); CLASSES = ['STAR','GALAXY','QSO']
RAW = BANDS + ['err_' + b for b in BANDS] + ['size_r', 'gal_l', 'gal_b']

# ---- choose the three CSV files: each one is looked for separately, and uploaded one at a time if missing ----
# Optional: set a path yourself to skip the search for that file, e.g. TRAIN_FILE = '/content/train_cosmic.csv'
TRAIN_FILE = TEST_FILE = SAMPLE_FILE = None

NEEDS = {'train':  RAW + ['class'],                 # columns each file must have (catches a file uploaded in the wrong box)
         'test':   RAW + ['object_id'],
         'sample': ['object_id'] + CLASSES}
LABELS = {'train':  'the TRAIN file (e.g. train_cosmic.csv)',
          'test':   'the TEST file (e.g. test_cosmos.csv)',
          'sample': 'the SAMPLE SUBMISSION file (e.g. sample_submission.csv)'}
EXPLICIT = {'train': TRAIN_FILE, 'test': TEST_FILE, 'sample': SAMPLE_FILE}

def pick(folder, key):
    """Best CSV in `folder` whose name contains `key`: exact '<key>.csv' first, otherwise the shortest name."""
    hits = [f for f in Path(folder).glob('*.csv') if key in f.name.lower()]
    if key != 'sample':
        hits = [f for f in hits if 'submission' not in f.name.lower()]      # never mistake your own output for data
    hits.sort(key=lambda f: (f.name.lower() != key + '.csv', len(f.name), f.name))
    return hits[0] if hits else None

def missing_columns(path, role):
    return [c for c in NEEDS[role] if c not in pd.read_csv(path, nrows=0).columns]

def search_one(role):
    """Look for one file in the usual places. Returns a Path or None."""
    if EXPLICIT[role]:
        return Path(EXPLICIT[role])
    folders = [Path(DRIVE_DIR)] if DRIVE_DIR else []
    folders += [Path('.'), Path('/content'), Path('/content/data'), Path('data')]
    if Path('/kaggle/input').exists():
        folders += sorted({p.parent for p in Path('/kaggle/input').rglob('*.csv')})
    for folder in folders:
        if folder.exists() and (f := pick(folder, role)):
            return f
    return None

def upload_one(role):
    """Colab only: opens the upload box for ONE file and asks again until it has the right columns."""
    from google.colab import files
    while True:
        print(f'\n>>> Upload {LABELS[role]} now. Choose just one file.')
        names = list(files.upload())
        if not names:
            print('No file was selected, try again.'); continue
        if len(names) > 1:
            print('You selected several files; using the first one:', names[0])
        path = Path(names[0])
        if not path.exists():                                   # Colab may rename a re-upload, e.g. "train (1).csv"
            path = max(Path('.').glob('*.csv'), key=lambda f: f.stat().st_mtime)
        bad = missing_columns(path, role)
        if not bad:
            return path
        print(f'{path.name} does not look like {LABELS[role]} (missing columns: {bad[:4]}). Please upload the right file.')

paths = {}
for role in ('train', 'test', 'sample'):
    p = search_one(role)
    if p is None:
        if 'google.colab' not in sys.modules:
            raise FileNotFoundError(f'Could not find the {role} CSV. Put it in the working folder or set '
                                    f'{role.upper()}_FILE at the top of this block.')
        p = upload_one(role)
    assert p.exists(), f'{role} file not found: {p}'
    bad = missing_columns(p, role)
    assert not bad, f'{p.name} cannot be the {role} file, it has no columns {bad[:4]}'
    paths[role] = p
    print(f'{role:6s} -> {p.resolve()}')                        # check these are the files you meant
data_path = paths['train'].resolve().parent
train  = pd.read_csv(paths['train'], usecols=RAW + ['class'])        # redshift is never read
test   = pd.read_csv(paths['test'], usecols=RAW + ['object_id'])
sample = pd.read_csv(paths['sample'])
if QUICK:
    train, _ = train_test_split(train, train_size=3000, stratify=train['class'], random_state=SEED)
    train = train.reset_index(drop=True)
y = train['class'].map({c: i for i, c in enumerate(CLASSES)}).to_numpy()
prior = np.bincount(y, minlength=3) / len(y)
print(f'{data_path} | train {len(train):,} | test {len(test):,} | class shares', prior.round(4))


## 2. Features
Row-by-row only (nothing is fitted on the test set). The most useful ones in a quick test were size x colour interactions (`size_color_*`), surface brightness (`mu_*`), and colours.

In [ ]:
def features(d):
    """Row-local features only: no fitted transforms, target encoding or test fitting.

    Magnitude errors are NOT flux errors. Flux SNR ~= 1.0857 / sigma_mag
    only in the high-SNR regime. Surface-brightness proxy assumes size is an
    angular radius; the unknown additive geometry constant is irrelevant.
    Derivative errors below account for shared-band covariance explicitly.
    """
    x = d[RAW].apply(pd.to_numeric, errors='coerce').copy()
    x = x.replace([np.inf, -np.inf], np.nan)
    m = x[BANDS]
    e = x[['err_' + b for b in BANDS]].copy()
    e.columns = BANDS
    invalid_err = (e <= 0) | e.isna()
    e = e.mask(e <= 0)
    safe_e = e.clip(lower=1e-5)
    size = x.size_r.where(x.size_r > 0)
    ls = np.log10(size)
    extra = {
        'size_invalid': size.isna().astype(float),
        'log_size': ls,
        'mag_mean': m.mean(axis=1),
        'mag_span': m.max(axis=1) - m.min(axis=1),
        'mag_std': m.std(axis=1),
        'err_mean': e.mean(axis=1),
        'err_max': e.max(axis=1),
        'err_rms': np.sqrt((e**2).mean(axis=1)),
        'invalid_err_count': invalid_err.sum(axis=1)
    }
    for b in BANDS:
        extra['log_err_' + b] = np.log10(safe_e[b])
        extra['flux_snr_proxy_' + b] = 1.085736 / safe_e[b]
        extra['mu_' + b] = m[b] + 5 * ls
        extra['centered_' + b] = m[b] - m.mean(axis=1)
    for a, b in itertools.combinations(BANDS, 2):
        c = m[a] - m[b]
        ce = np.sqrt(safe_e[a]**2 + safe_e[b]**2)
        name = a + '_' + b
        extra['color_' + name] = c
        extra['color_err_' + name] = ce
        extra['color_significance_' + name] = (c / ce).clip(-1e4, 1e4)
        extra['log_err_ratio_' + name] = np.log(safe_e[a] / safe_e[b])
        # Relative flux ratio; clipping prevents catastrophic exponent overflow.
        extra['flux_ratio_' + name] = 10. ** np.clip(-0.4 * c, -6, 6)
        extra['size_color_' + name] = ls * c
    for a, b, c in zip(BANDS[:-2], BANDS[1:-1], BANDS[2:]):
        curv = m[a] - 2 * m[b] + m[c]
        # Var(a - 2b + c), not Var(a-b)+Var(b-c).
        sigma = np.sqrt(safe_e[a]**2 + 4 * safe_e[b]**2 + safe_e[c]**2)
        key = a + b + c
        extra['curvature_' + key] = curv
        extra['curvature_significance_' + key] = (curv / sigma).clip(-1e4, 1e4)
        # Bounded analogue of unstable adjacent-colour division.
        ca, cb = m[a] - m[b], m[b] - m[c]
        extra['color_angle_' + key] = np.arctan2(ca, cb)
    # SDSS-style principal colour projections: fixed photometric combinations,
    # not learned from any outside training catalogue. Useful stellar-locus axes.
    extra['locus_s'] = -.249 * m.u + .794 * m.g - .555 * m.r + .234
    extra['locus_w'] = -.227 * m.g + .792 * m.r - .567 * m.i + .050
    extra['locus_x'] = .707 * m.g - .707 * m.r - .988
    extra['locus_y'] = -.270 * m.r + .800 * m.i - .534 * m.z + .054
    lon, lat = np.deg2rad(x.gal_l), np.deg2rad(x.gal_b)
    extra.update(
        abs_b=x.gal_b.abs(),
        sin_l=np.sin(lon), cos_l=np.cos(lon),
        sin_b=np.sin(lat), cos_b=np.cos(lat),
        gal_x=np.cos(lat) * np.cos(lon),
        gal_y=np.cos(lat) * np.sin(lon),
        abs_sin_b=np.abs(np.sin(lat))
    )
    # Remove discontinuous raw longitude; sin_b is the third unit-vector axis.
    x = x.drop(columns=['gal_l'])
    x = pd.concat([x, pd.DataFrame(extra, index=d.index)], axis=1)
    return x.replace([np.inf, -np.inf], np.nan).astype('float32')



X, Xt = features(train), features(test)
assert list(X) == list(Xt) and not any('redshift' in c for c in X)
print('features:', X.shape[1])

## 2b. v6 extra features (local blur, error fuzziness)
Uses only unlabelled columns of train + test pooled. No class label is touched.

In [ ]:
# ---------- v6 extra features (run AFTER the features() cell) ----------
# Both groups use ONLY the unlabelled columns (magnitudes, errors, size, sky position) of train + test pooled together.
# No class label is ever touched, so there is no label leakage; test rows just act as additional sky neighbours.
K_NB = 64

def sky_neighbours(pool, K=K_NB):
    """K nearest neighbours of every object on the sky (itself excluded). Returns (idx, dist) of shape n x K."""
    lon, lat = np.deg2rad(pool.gal_l.to_numpy(float)), np.deg2rad(pool.gal_b.to_numpy(float))
    xyz = np.c_[np.cos(lat) * np.cos(lon), np.cos(lat) * np.sin(lon), np.sin(lat)]
    dist, idx = cKDTree(xyz).query(xyz, k=K + 1, workers=-1)
    n = len(pool)
    hit = idx == np.arange(n)[:, None]
    hit[~hit.any(axis=1), -1] = True              # self not in the list (ties at identical positions): drop the farthest instead
    return idx[~hit].reshape(n, K), dist[~hit].reshape(n, K)

def row_quantile(a, q, min_count=3):
    """Quantile of every row, ignoring NaN (fast: sort, then pick by position)."""
    s = np.sort(a, axis=1)                         # NaN goes to the end
    cnt = np.isfinite(a).sum(axis=1)
    pos = np.clip(np.round(q * (cnt - 1)).astype(int), 0, a.shape[1] - 1)
    out = s[np.arange(len(a)), pos]
    out[cnt < min_count] = np.nan
    return out

def blur_features(pool, idx, dist):
    """Stars' sizes follow the local seeing; galaxies' do not. Compare each object with the sizes around it."""
    size = pool.size_r.to_numpy(float); size = np.where(size > 0, size, np.nan)
    ls = np.log10(size)
    nb = ls[idx]                                   # n x K log10 sizes of the neighbours
    pointlike = np.where(size[idx] < 2.0, nb, np.nan)   # neighbours that look like point sources (size < 2: 94% non-galaxies)
    f = {'nb_med8': row_quantile(nb[:, :8], .5), 'nb_med32': row_quantile(nb[:, :32], .5),
         'nb_med64': row_quantile(nb, .5), 'nb_q20_32': row_quantile(nb[:, :32], .2),
         'nb_psf64': row_quantile(pointlike, .5), 'nb_psf64_q25': row_quantile(pointlike, .25),
         'nb_pointlike_frac': np.isfinite(pointlike).mean(axis=1),
         'nb_logdist8': np.log10(dist[:, 7] + 1e-9)}
    for k in ('nb_med32', 'nb_q20_32', 'nb_psf64'):
        f['own_minus_' + k] = ls - f[k]
    return f

def fuzz_features(pool, idx):
    """How 'fuzzy' an object is given its measurement errors.
    A point source of a given magnitude has a smaller magnitude error than an extended one (bigger aperture, more sky noise).
    resid = log10(error) minus the typical log10(error) at the same magnitude (binned median over train+test, no labels).
    The 'local' version also removes what the neighbouring objects show (sky brightness / seeing of that field)."""
    f, resid = {}, {}
    for b in BANDS:
        mag = pool[b].to_numpy(float); le = np.log10(np.clip(pool['err_' + b].to_numpy(float), 1e-5, None))
        ok = np.isfinite(mag) & np.isfinite(le)
        edges = np.unique(np.quantile(mag[ok], np.linspace(0, 1, 41)))
        which = np.clip(np.digitize(mag[ok], edges[1:-1]), 0, len(edges) - 2)
        cm = np.array([np.median(mag[ok][which == j]) for j in range(len(edges) - 1) if (which == j).any()])
        cl = np.array([np.median(le[ok][which == j]) for j in range(len(edges) - 1) if (which == j).any()])
        resid[b] = le - np.interp(mag, cm, cl)
        f['fz_resid_' + b] = resid[b]
    f['fz_mean_gri'] = np.mean([resid[b] for b in 'gri'], axis=0)
    for b in 'gri':
        f['fz_local_' + b] = resid[b] - row_quantile(resid[b][idx][:, :32], .5)
    return f

def add_extra(X, Xt):
    pool = pd.concat([train[RAW], test[RAW]], ignore_index=True)
    idx, dist = sky_neighbours(pool)
    extra = {}
    if USE_BLUR: extra.update(blur_features(pool, idx, dist))
    if USE_FUZZ: extra.update(fuzz_features(pool, idx))
    E = pd.DataFrame(extra).astype('float32').replace([np.inf, -np.inf], np.nan)
    n = len(train)
    E_tr, E_te = E.iloc[:n].copy(), E.iloc[n:].copy()
    E_tr.index, E_te.index = X.index, Xt.index
    return pd.concat([X, E_tr], axis=1), pd.concat([Xt, E_te], axis=1), list(E.columns)

t0 = time.time()
BASE_COLS = list(X)
if USE_BLUR or USE_FUZZ:
    X, Xt, NEW_COLS = add_extra(X, Xt)
else:
    NEW_COLS = []
BLUR_COLS = [c for c in NEW_COLS if c.startswith(('nb_', 'own_minus_'))]
FUZZ_COLS = [c for c in NEW_COLS if c.startswith('fz_')]
assert list(X) == list(Xt) and not any('redshift' in c for c in X)
print(f'features: {len(BASE_COLS)} base + {len(NEW_COLS)} new = {X.shape[1]}  ({time.time() - t0:.0f}s)')
print('NaN share in new columns (train):', X[NEW_COLS].isna().mean().round(3).to_dict() if NEW_COLS else '-')

## 3. Scoring and calibration helpers

In [ ]:
def normalize(p):
    p = np.maximum(np.asarray(p, dtype=float), 1e-15)
    return p / p.sum(axis=1, keepdims=True)

def loss(y_true, p, metric=None):
    metric = metric or METRIC
    v = -np.log(normalize(p)[np.arange(len(y_true)), y_true])
    if metric == 'plain':
        return float(v.mean())
    return float(np.mean([v[y_true == k].mean() for k in range(3)]))   # class-balanced

def shape(p):
    # Unweighted models estimate the true class probabilities, which is what plain log loss wants.
    # The class-balanced metric wants them divided by the class share (then renormalised).
    p = normalize(p)
    return p if METRIC == 'plain' else normalize(p / prior)

def temper(p, t):
    q = np.exp(np.log(np.maximum(p, 1e-15)) / t)
    return q / q.sum(axis=1, keepdims=True)

def smooth(p, floor):
    return (1 - 3 * floor) * p + floor          # every probability stays >= floor

## 3b. Neural-net inputs and model

In [ ]:
# ---------- neural net v2: piecewise-linear numeric embeddings + LayerNorm + AdamW(wd 1e-2) + EMA ----------
# Inputs: rank-gauss transform fitted on train + test FEATURES only (no labels).
if USE_NN:
    allX = pd.concat([X, Xt], ignore_index=True)
    qt = QuantileTransformer(n_quantiles=min(1000, len(allX)), output_distribution='normal',
                             subsample=200_000, random_state=SEED)
    Zall = qt.fit_transform(allX.to_numpy(np.float64))
    Zall = np.clip(np.nan_to_num(Zall, nan=0.0), -5, 5)
    keep = Zall.std(axis=0) > 1e-6                                 # drop constant columns
    Zall = Zall[:, keep].astype('float32')
    Z, Zt = Zall[:len(X)], Zall[len(X):]
    # bin edges for the piecewise-linear encoding: quantiles of each (rank-gauss) feature, label-free
    edges = np.quantile(Zall, np.linspace(0, 1, NN_BINS + 1), axis=0).T.astype('float32')   # F x (T+1)
    PLE_LO = torch.from_numpy(edges[:, :-1].copy()).to(DEVICE)
    PLE_W  = torch.from_numpy(np.maximum(edges[:, 1:] - edges[:, :-1], 1e-3)).to(DEVICE)       # F x T
    ZD, ZtD = torch.from_numpy(Z).to(DEVICE), torch.from_numpy(Zt).to(DEVICE)
    yD = torch.from_numpy(np.asarray(y)).long().to(DEVICE)
    print(f'NN inputs: {Z.shape} | dropped constant columns: {int((~keep).sum())} | device: {DEVICE} | '
          f'bins {NN_BINS} | embedding dim {NN_EMB} | seeds {N_SEEDS_NN}')

    # ---- model definitions (only when the net is switched on) ----
    class NumEmbed(nn.Module):
        """Every numeric feature gets its own small embedding: piecewise-linear encoding over quantile bins,
        then a per-feature linear map (T -> d) and ReLU. Output is flattened to F*d."""
        def __init__(self, n_feat, n_bins, d, lo, width):
            super().__init__()
            self.register_buffer('lo', lo); self.register_buffer('width', width)
            s = n_bins ** -0.5
            self.W = nn.Parameter(torch.empty(n_feat, n_bins, d).uniform_(-s, s))
            self.b = nn.Parameter(torch.empty(n_feat, d).uniform_(-s, s))
        def forward(self, x):                                              # x: B x F
            e = (x.unsqueeze(2) - self.lo) / self.width                    # B x F x T
            e = torch.cat([e[..., :1].clamp(-5.0, 1.0),                    # first bin: open below
                           e[..., 1:-1].clamp(0.0, 1.0),
                           e[..., -1:].clamp(0.0, 5.0)], dim=2)            # last bin: open above
            return F.relu(torch.einsum('bft,ftd->bfd', e, self.W) + self.b).flatten(1)

    class TabNet(nn.Module):
        def __init__(self, n_feat):
            super().__init__()
            self.emb = NumEmbed(n_feat, NN_BINS, NN_EMB, PLE_LO, PLE_W)
            din, layers = n_feat * NN_EMB + n_feat, []                     # embeddings + the raw rank-gauss values
            for w, p in zip(NN_WIDTHS, NN_DROPS):
                layers += [nn.Linear(din, w), nn.LayerNorm(w), nn.SiLU(), nn.Dropout(p)]; din = w
            self.body, self.head = nn.Sequential(*layers), nn.Linear(din, 3)
        def forward(self, x):
            return self.head(self.body(torch.cat([self.emb(x), x], dim=1)))

    @torch.no_grad()
    def predict_logits(model, x, chunk=8192):
        model.eval()
        return torch.cat([model(x[i:i + chunk]) for i in range(0, len(x), chunk)])

    def fit_mlp(tr, va, seed):
        torch.manual_seed(seed); np.random.seed(seed)
        verbose = (seed - SEED) < 1000                                     # print the epoch log for seed 0 only
        tr_t, va_t = torch.from_numpy(tr).to(DEVICE), torch.from_numpy(va).to(DEVICE)
        xv, yv = ZD[va_t], yD[va_t]
        net = TabNet(ZD.shape[1]).to(DEVICE)
        ema = copy.deepcopy(net).eval()
        for p in ema.parameters(): p.requires_grad_(False)
        # weight decay on weight matrices only (not on biases, LayerNorm scales, or embedding biases)
        decay   = [p for n, p in net.named_parameters() if p.ndim >= 2 and not n.endswith('.b')]
        nodecay = [p for n, p in net.named_parameters() if not (p.ndim >= 2 and not n.endswith('.b'))]
        opt = torch.optim.AdamW([{'params': decay, 'weight_decay': NN_WD}, {'params': nodecay, 'weight_decay': 0.0}], lr=NN_LR)
        n = len(tr); steps = (n + NN_BATCH - 1) // NN_BATCH
        sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=NN_LR, total_steps=NN_EPOCHS * steps, pct_start=0.2)
        t, best, best_ep = 0, 1e9, 0
        for ep in range(NN_EPOCHS):
            net.train(); perm = torch.randperm(n, device=DEVICE); tl = torch.zeros((), device=DEVICE)
            for b in range(0, n, NN_BATCH):
                ii = tr_t[perm[b:b + NN_BATCH]]
                loss_b = F.cross_entropy(net(ZD[ii]), yD[ii])
                opt.zero_grad(set_to_none=True); loss_b.backward()
                nn.utils.clip_grad_norm_(net.parameters(), 5.0); opt.step(); sched.step()
                tl += loss_b.detach() * len(ii); t += 1
                d = min(NN_EMA, (1 + t) / (10 + t))                        # EMA with warm-up
                with torch.no_grad():
                    for pe, p in zip(ema.parameters(), net.parameters()):
                        pe.mul_(d).add_(p.detach(), alpha=1 - d)
            lv = F.cross_entropy(predict_logits(ema, xv), yv).item()      # validation loss of the EMA weights
            if lv < best: best, best_ep = lv, ep + 1
            if verbose and ((ep + 1) % 5 == 0 or ep == 0 or ep + 1 == NN_EPOCHS):
                print(f'      MLP ep {ep + 1:2d}/{NN_EPOCHS}  train {tl.item() / n:.4f} (dropout on)  val(EMA) {lv:.4f}', flush=True)
        print(f'      MLP final val {lv:.4f} | best val {best:.4f} at epoch {best_ep}/{NN_EPOCHS}'
              f'{"  <- best epoch is early: raise dropout / weight decay" if best_ep < 0.6 * NN_EPOCHS else ""}', flush=True)
        pv = F.softmax(predict_logits(ema, xv), 1).cpu().numpy()
        pt = F.softmax(predict_logits(ema, ZtD), 1).cpu().numpy()
        return pv, pt, NN_EPOCHS

## 4. Train the models on the same 5 folds (with seed bagging)
Each fold hides 20% of train, trains on the rest and predicts the hidden part (out-of-fold, OOF). Every model is trained `N_SEEDS` times with different seeds and the predictions are averaged. Finished (fold, model, seed) runs are saved in `ckpt/` and skipped on a re-run.

In [ ]:
rounds, patience = (60, 15) if QUICK else (4000, 150)
folds = list(StratifiedKFold(FOLDS, shuffle=True, random_state=SEED).split(X, y))   # fixed split: seeds never change it
NAMES = ['LightGBM', 'XGBoost'] + (['CatBoost'] if USE_CATBOOST else []) + (['MLP'] if USE_NN else [])
SEEDS_OF = {'LightGBM': N_SEEDS, 'XGBoost': N_SEEDS, 'CatBoost': N_SEEDS, 'MLP': N_SEEDS_NN}
M = len(NAMES)
oof   = np.zeros((len(y), M, 3))            # rows x models x classes, seed-averaged
oof0  = np.zeros((len(y), M, 3))            # same, but seed 0 only (to show what bagging adds)
tpred = np.zeros((len(test), M, 3))
records = []
tag = TAG + ('_quick' if QUICK else '_full')

def fit_one(name, tr, va, seed):
    """Train one model with one seed. Returns (val probs, test probs, number of trees / epochs)."""
    if name == 'MLP':
        return fit_mlp(tr, va, seed)
    if name == 'LightGBM':
        params = dict(objective='multiclass', num_class=3, metric='multi_logloss', learning_rate=.05,
                      num_leaves=31, min_data_in_leaf=50, feature_fraction=.7, bagging_fraction=.85,
                      bagging_freq=1, lambda_l2=5., verbosity=-1, num_threads=THREADS, seed=seed)
        m = lgb.train(params, lgb.Dataset(X.iloc[tr], y[tr]), rounds,
                      valid_sets=[lgb.Dataset(X.iloc[va], y[va])],
                      callbacks=[lgb.early_stopping(patience, verbose=False)])
        n = m.best_iteration
        return m.predict(X.iloc[va], num_iteration=n), m.predict(Xt, num_iteration=n), n
    if name == 'XGBoost':
        params = dict(objective='multi:softprob', num_class=3, eval_metric='mlogloss', tree_method='hist',
                      device='cuda' if GPU else 'cpu', eta=.05, max_depth=6, min_child_weight=10,
                      subsample=.85, colsample_bytree=.7, reg_lambda=6., seed=seed, nthread=THREADS)
        dt, dv = xgb.DMatrix(X.iloc[tr], label=y[tr]), xgb.DMatrix(X.iloc[va], label=y[va])
        m = xgb.train(params, dt, rounds, evals=[(dv, 'valid')], early_stopping_rounds=patience, verbose_eval=False)
        n = m.best_iteration + 1
        return m.predict(dv, iteration_range=(0, n)), m.predict(xgb.DMatrix(Xt), iteration_range=(0, n)), n
    params = dict(loss_function='MultiClass', learning_rate=.08, depth=7, l2_leaf_reg=7.,
                  iterations=rounds, random_seed=seed, thread_count=THREADS, verbose=0,
                  early_stopping_rounds=patience, allow_writing_files=False,
                  task_type='GPU' if GPU else 'CPU')
    if not GPU:
        params['rsm'] = .5                       # feature sub-sampling (CPU only): faster
    m = CatBoostClassifier(**params)
    xc, xv, xx = (z.fillna(-999.) for z in (X.iloc[tr], X.iloc[va], Xt))
    m.fit(Pool(xc, y[tr]), eval_set=Pool(xv, y[va]), use_best_model=True)
    assert np.array_equal(m.classes_, np.arange(3))
    return m.predict_proba(xv), m.predict_proba(xx), m.tree_count_

t_start = time.time()
for f, (tr, va) in enumerate(folds):
    for j, name in enumerate(NAMES):
        ns = SEEDS_OF[name]
        pvs, pts = [], []
        for s in range(ns):
            seed = SEED + 97 * f + 1000 * s
            ck = CKPT_DIR / f'{tag}_{NN_VERSION if name == "MLP" else name}_{f}_s{s}.npz'
            if ck.exists():
                z = np.load(ck); pv, pt = z['pv'], z['pt']
                n = int(z['n']) if 'n' in z.files else -1
                secs = 0; loaded = True
            else:
                loaded = False
                t0 = time.time()
                pv, pt, n = fit_one(name, tr, va, seed)
                pv, pt = normalize(pv), normalize(pt)
                secs = round(time.time() - t0)
                np.savez_compressed(ck, pv=pv, pt=pt, n=n)
            pvs.append(pv); pts.append(pt)
            records.append(dict(fold=f+1, model=name, seed_idx=s, trees=int(n), plain=loss(y[va], pv, 'plain'),
                                balanced=loss(y[va], pv, 'balanced'), seconds=secs))
            print(f"fold {f+1}/{FOLDS} {name} seed {s+1}/{ns}: plain {records[-1]['plain']:.4f} | "
                  f"{'epochs' if name == 'MLP' else 'trees'} {n} | {'checkpoint' if loaded else str(secs) + 's'}", flush=True)
        pv_bag, pt_bag = normalize(np.mean(pvs, axis=0)), normalize(np.mean(pts, axis=0))
        oof[va, j] = pv_bag; oof0[va, j] = pvs[0]; tpred[:, j] += pt_bag / FOLDS
        if ns > 1:
            print(f"   -> fold {f+1} {name}: seed 0 only {loss(y[va], pvs[0], 'plain'):.4f}  |  "
                  f"bagged x{ns} {loss(y[va], pv_bag, 'plain'):.4f}", flush=True)
print(f'training finished in {(time.time() - t_start) / 60:.1f} min (runs loaded from checkpoints count as 0)')

## 5. Blend, calibrate, compare
Blend weights are fitted on the OOF predictions (3 numbers, so there is almost no risk of overfitting). A single temperature then softens or sharpens the blend.

In [ ]:
def blend(p3, w):
    return normalize(np.einsum('nmc,m->nc', p3, w))

obj = lambda w: loss(y, shape(blend(oof, np.abs(w) / np.abs(w).sum())))
fit = minimize(obj, np.ones(M) / M, method='Nelder-Mead', options={'xatol': 1e-3, 'fatol': 1e-7, 'maxiter': 200})
W = np.abs(fit.x) / np.abs(fit.x).sum()
pb = shape(blend(oof, W))

T = minimize_scalar(lambda t: loss(y, temper(pb, t)), bounds=(.7, 1.5), method='bounded').x
pbt = temper(pb, T)

FLOORS = [1e-5, 1e-4, 3e-4, 1e-3]
FLOOR = min(FLOORS, key=lambda fl: loss(y, smooth(pbt, fl)))
pfin = smooth(pbt, FLOOR)

rows = [dict(model=n, plain=loss(y, oof[:, j], 'plain'), balanced=loss(y, oof[:, j], 'balanced'),
             accuracy=float((oof[:, j].argmax(1) == y).mean())) for j, n in enumerate(NAMES)]
if N_SEEDS > 1:                      # what bagging added, on the same folds
    rows += [dict(model=n + ' (seed 0 only)', plain=loss(y, oof0[:, j], 'plain'), balanced=loss(y, oof0[:, j], 'balanced'),
                  accuracy=float((oof0[:, j].argmax(1) == y).mean())) for j, n in enumerate(NAMES)]
    p0 = shape(blend(oof0, W))
    rows.append(dict(model='Blend, seed 0 only (same weights)', plain=loss(y, p0, 'plain'),
                     balanced=loss(y, p0, 'balanced'), accuracy=float((p0.argmax(1) == y).mean())))
for label, p in [('Blend', pb), ('Blend + temperature', pbt), (f'FINAL (floor {FLOOR:g})', pfin)]:
    rows.append(dict(model=label, plain=loss(y, p, 'plain'), balanced=loss(y, p, 'balanced'),
                     accuracy=float((p.argmax(1) == y).mean())))
summary = pd.DataFrame(rows).set_index('model')
print('Blend weights', {k: round(float(v), 3) for k, v in zip(NAMES, W)}, '| temperature', round(T, 3), '| floor', FLOOR)
print(f'\nMETRIC = {METRIC}  (the number to compare with the leaderboard is the "{METRIC}" column) | N_SEEDS = {N_SEEDS}')
print(summary.round(4).to_string())

# honest noise estimate: score of the final blend on each fold
fs = [loss(y[va], pfin[va]) for _, va in folds]
print('\nFinal blend by fold:', np.round(fs, 4), '| std', round(float(np.std(fs, ddof=1)), 4))
summary.to_csv('cv_summary.csv'); pd.DataFrame(records).to_csv('fold_scores.csv', index=False)

### v5 vs v6 on the same 5 folds

In [ ]:
# ---------- compare with the v5 bagging run on the SAME 5 folds (numbers from your v5 printout) ----------
V5_FOLDS = np.array([0.1564, 0.1569, 0.1597, 0.1503, 0.1521]); V5_CV = 0.1551
if not QUICK and METRIC == 'plain' and FOLDS == 5 and SEED == 42:
    v6 = np.array(fs); d = V5_FOLDS - v6
    print(f'\nv5 CV {V5_CV:.4f}  ->  v6 CV {loss(y, pfin):.4f}   (gain {V5_CV - loss(y, pfin):+.4f})')
    print('per-fold gain (v5 minus v6, positive = v6 better):', np.round(d, 4),
          f'| better in {int((d > 0).sum())} of 5 folds | mean {d.mean():+.4f} +- {d.std(ddof=1) / np.sqrt(5):.4f}')
    print('Trust the new features only if v6 is better in (almost) every fold. Public score may differ by +-0.006.')

## 6. Write submission.csv

In [ ]:
final = smooth(temper(shape(blend(tpred, W)), T), FLOOR)

if QUICK:
    print('QUICK mode: code check only, no submission written.')
else:
    sub = pd.DataFrame(final, columns=CLASSES, index=test.object_id).reindex(sample.object_id).reset_index()
    assert list(sub.columns) == list(sample.columns)
    assert np.array_equal(sub.object_id.to_numpy(), sample.object_id.to_numpy())
    assert np.isfinite(sub[CLASSES].to_numpy()).all() and np.allclose(sub[CLASSES].sum(axis=1), 1)
    assert (sub[CLASSES].to_numpy() >= FLOOR - 1e-12).all()
    sub.to_csv('submission_v7.csv', index=False)
    print('Saved submission_v7.csv', sub.shape, '| predicted shares', sub[CLASSES].to_numpy().mean(0).round(4), '| train shares', prior.round(4))
    if 'google.colab' in sys.modules:
        from google.colab import files
        files.download('submission_v7.csv')

## 7. Optional: feature-group check (set `RUN_ABLATION = True` in the first cell)

In [ ]:
# ---------- OPTIONAL (RUN_ABLATION = True): which feature group helps, on all 5 folds, paired ----------
if RUN_ABLATION:
    variants = {'v5 base': BASE_COLS, '+ blur': BASE_COLS + BLUR_COLS, '+ fuzz': BASE_COLS + FUZZ_COLS,
                '+ both': BASE_COLS + BLUR_COLS + FUZZ_COLS}
    res = {k: [] for k in variants}
    for k, cols in variants.items():
        for f, (tr, va) in enumerate(folds):
            p = dict(objective='multiclass', num_class=3, metric='multi_logloss', learning_rate=.1, num_leaves=31,
                     min_data_in_leaf=50, feature_fraction=.7, bagging_fraction=.85, bagging_freq=1, lambda_l2=5.,
                     verbosity=-1, num_threads=THREADS, seed=SEED + 97 * f)
            m = lgb.train(p, lgb.Dataset(X.iloc[tr][cols], y[tr]), 60 if QUICK else 1500,
                          valid_sets=[lgb.Dataset(X.iloc[va][cols], y[va])],
                          callbacks=[lgb.early_stopping(15 if QUICK else 50, verbose=False)])
            res[k].append(loss(y[va], m.predict(X.iloc[va][cols], num_iteration=m.best_iteration), 'plain'))
        print(f'{k:8s} mean {np.mean(res[k]):.4f}  folds {np.round(res[k], 4)}', flush=True)
    for k in list(variants)[1:]:
        d = np.array(res['v5 base']) - np.array(res[k])
        print(f'{k:8s} gain {d.mean():+.4f} +- {d.std(ddof=1) / np.sqrt(len(d)):.4f} | better in {int((d > 0).sum())}/{len(d)} folds')